Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

Chargement du dataset

In [ ]:
df = pd.read_csv("C:/Users/COD/Desktop/projet_house_rent/House_Rent_Dataset.csv")

In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
# Aperçu des valeurs uniques par colonne catégorielle
cat_cols = ['City', 'Furnishing Status', 'Area Type', 'Tenant Preferred', 'Point of Contact']
for col in cat_cols:
    vals = df[col].unique()
    print(f"{col:25s} → {len(vals)} valeurs : {list(vals)}")

Exploration initiale

In [ ]:
# Statistiques descriptives des colonnes numériques
df[['Rent', 'Size', 'BHK', 'Bathroom']].describe().round(0)

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
print("Valeurs manquantes par colonne :")
print(missing[missing > 0] if missing.sum() > 0 else "  ➜ Aucune valeur manquante !")

In [ ]:
# Doublons
n_dup = df.duplicated().sum()
print(f"Doublons : {n_dup}")

In [ ]:
# Distribution du loyer AVANT nettoyage
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Rent'], bins=60, color='#4F8EF7', edgecolor='white')
axes[0].set_title('Distribution du loyer (brut)', fontsize=13)
axes[0].set_xlabel('Loyer (₹/mois)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

axes[1].boxplot(df['Rent'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='#4F8EF7', color='navy'))
axes[1].set_title('Boxplot du loyer (brut)', fontsize=13)
axes[1].set_xlabel('Loyer (₹/mois)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

plt.tight_layout()
plt.show()

print(f"Max loyer brut : ₹{df['Rent'].max():,}")
print(f"Min loyer brut : ₹{df['Rent'].min():,}")

Nettoyage 

In [ ]:
# ── 4.1 Suppression des doublons ─────────────────────────────
nb_avant = len(df)
df = df.drop_duplicates()
print(f"Doublons supprimés : {nb_avant - len(df)}")

In [ ]:
# ── 4.2 Valeurs manquantes ────────────────────────────────────
# Supprimer les lignes sans loyer, taille, BHK ou ville
df = df.dropna(subset=['Rent', 'Size', 'BHK', 'City'])

# Remplir les colonnes texte par le mode
for col in ['Furnishing Status', 'Tenant Preferred', 'Area Type']:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  {col} : valeurs manquantes remplacées par '{mode_val}'")

print("✅ Valeurs manquantes traitées")

In [ ]:
# ── 4.3 Correction des types ──────────────────────────────────
df['Posted On'] = pd.to_datetime(df['Posted On'], errors='coerce')
df['BHK']       = df['BHK'].astype(int)
df['Bathroom']  = df['Bathroom'].astype(int)
df['Rent']      = df['Rent'].astype(int)
df['Size']      = df['Size'].astype(int)

print("Types après correction :")
print(df[['Posted On', 'BHK', 'Bathroom', 'Rent', 'Size']].dtypes)

In [ ]:
# ── 4.4 Suppression des outliers ─────────────────────────────
# Méthode IQR (intervalle interquartile × 3)
nb_avant = len(df)

Q1 = df['Rent'].quantile(0.25)
Q3 = df['Rent'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 3 * IQR
upper = Q3 + 3 * IQR

df = df[(df['Rent'] >= lower) & (df['Rent'] <= upper)]
df = df[df['Size'] >= 50]           # Minimum 50 sq ft
df = df[(df['BHK'] >= 1) & (df['BHK'] <= 10)]

df = df.reset_index(drop=True)

print(f"Lignes supprimées (outliers) : {nb_avant - len(df)}")
print(f"Lignes conservées            : {len(df)}")
print(f"Loyer max après nettoyage    : ₹{df['Rent'].max():,}")

In [ ]:
# Comparaison avant / après nettoyage
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Rent'], bins=60, color='#F4845F', edgecolor='white', alpha=0.8)
axes[0].set_title(f'AVANT nettoyage  ({len(df):,} lignes)', fontsize=12)
axes[0].set_xlabel('Loyer (₹/mois)')

axes[1].hist(df['Rent'], bins=60, color='#4F8EF7', edgecolor='white', alpha=0.8)
axes[1].set_title(f'APRÈS nettoyage  ({len(df):,} lignes)', fontsize=12)
axes[1].set_xlabel('Loyer (₹/mois)')

plt.tight_layout()
plt.show()

Feature Engineering

In [ ]:
# Fonctions utilitaires pour extraire l'étage
def extract_floor_number(floor_str):
    try:
        part = str(floor_str).split('out')[0].strip().lower()
        if part in ('ground', 'lower basement', 'upper basement'):
            return 0
        return int(part)
    except (ValueError, IndexError):
        return -1

def extract_total_floors(floor_str):
    try:
        parts = str(floor_str).split('out of')
        return int(parts[1].strip()) if len(parts) == 2 else -1
    except (ValueError, IndexError):
        return -1

# Application
df['rent_per_sqft'] = (df['Rent'] / df['Size']).round(2)
df['floor_number']  = df['Floor'].apply(extract_floor_number)
df['total_floors']  = df['Floor'].apply(extract_total_floors)
df['is_ground']     = (df['floor_number'] == 0).astype(int)
df['month_posted']  = df['Posted On'].dt.month

print("Nouvelles colonnes créées :")
new_cols = ['rent_per_sqft', 'floor_number', 'total_floors', 'is_ground', 'month_posted']
print(df[new_cols].head(5).to_string())

In [ ]:
# Vérification finale du DataFrame nettoyé
print("═" * 50)
print(f"  Dimensions finales : {df.shape}")
print(f"  Colonnes           : {df.shape[1]}")
print(f"  Valeurs nulles     : {df.isnull().sum().sum()}")
print(f"  Doublons           : {df.duplicated().sum()}")
print("═" * 50)
df.head(3)

Export du DataFrame propre

In [ ]:
df.to_csv("house_rent_clean.csv", index=False)
print("✅ Fichier sauvegardé : house_rent_clean.csv")
print(f"   {len(df):,} lignes × {len(df.columns)} colonnes")

In [ ]:
# 1. Définir le chemin complet pour le retrouver facilement sur votre bureau
chemin_sauvegarde = "C:/Users/COD/Desktop/projet_house_rent/house_rent_clean.csv"

# 2. Sauvegarder le fichier nettoyé
df.to_csv(chemin_sauvegarde, index=False)

# 3. Afficher la confirmation
print(f"✅ Fichier sauvegardé : {chemin_sauvegarde}")
print(f"   {len(df):,} lignes × {len(df.columns)} colonnes")